In [35]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/checking-logs.sqlite")

In [36]:
pd.io.sql.read_sql("PRAGMA table_info(test);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


In [37]:
pd.io.sql.read_sql("SELECT * FROM test LIMIT 10", conn)

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


In [38]:
pd.io.sql.read_sql("SELECT * FROM deadlines LIMIT 10", conn)

,index,labs,deadlines
0,0,laba04,1587945599
1,1,laba04s,1587945599
2,2,laba05,1588550399
3,4,laba06,1590364799
4,5,laba06s,1590364799
5,3,project1,1589673599


In [39]:
df_min = pd.io.sql.read_sql(
    """--sql
        WITH
        CTE AS (
        SELECT test.uid, (strftime ('%s', test.first_commit_ts) - deadlines) / 3600 AS min_delta
        FROM test JOIN deadlines 
        ON test.labname = deadlines.labs
        WHERE test.labname != 'project1')
        SELECT * FROM  CTE
        ORDER BY min_delta ASC LIMIT 1""",
    conn,
)
df_min

,uid,min_delta
0,user_30,-202


In [40]:
df_max = pd.io.sql.read_sql(
    """--sql
        WITH
        CTE AS (
        SELECT test.uid, (strftime ('%s', test.first_commit_ts) - deadlines) / 3600 AS max_delta
        FROM test JOIN deadlines 
        ON test.labname = deadlines.labs
        WHERE test.labname != 'project1')
        SELECT * FROM CTE
        ORDER BY max_delta DESC LIMIT 1""",
    conn,
)
df_max

,uid,max_delta
0,user_25,-2


In [41]:
df_avg = pd.io.sql.read_sql(
    """--sql
        WITH
        CTE AS (
        SELECT t.uid, labname,
        (strftime ('%s', t.first_commit_ts) - d.deadlines) / 3600 AS delta
        FROM test AS t JOIN deadlines AS d 
        ON t.labname = d.labs
        WHERE t.labname != 'project1')
        SELECT AVG(delta) AS avg_delta
        FROM CTE""",
    conn,
)
df_avg

,avg_delta
0,-89.125


In [42]:
views_diff = pd.io.sql.read_sql(
    """--sql
    WITH commits_cte AS (
    SELECT t.uid, AVG((strftime ('%s', t.first_commit_ts) - d.deadlines) / 3600) AS avg_diff
    FROM test AS t JOIN deadlines AS d 
    ON t.labname = d.labs
    WHERE t.labname != 'project1'
    GROUP BY t.uid
    ),
    views_cte AS (
    SELECT uid, COUNT(datetime) AS pageviews
    FROM pageviews 
    GROUP BY uid)

    SELECT c.uid, c.avg_diff, IFNULL(v.pageviews, 0) AS pageviews 
    FROM commits_cte AS c 
    LEFT JOIN views_cte AS v
    ON c.uid=v.uid
    """,
    conn,
)
views_diff

,uid,avg_diff,pageviews
0,user_1,-64.400000,28
1,user_10,-74.800000,89
2,user_14,-159.000000,143
3,user_17,-61.600000,47
4,user_18,-5.666667,3
5,user_19,-98.750000,16
6,user_21,-95.500000,10
7,user_25,-92.600000,179
8,user_28,-86.400000,149
9,user_3,-105.400000,317


In [ ]:
views_diff.corr(numeric_only=True)

,avg_diff,pageviews
avg_diff,1.000000,-0.279736
pageviews,-0.279736,1.000000


In [ ]:
# conn.close()